# Identificación y autoría

**Speaker:** Eduardo Osorio Venegas

**Apoyo de inteligencia artificial generativa:** ChatGPT (OpenAI), para la elaboración de las celdas Markdown.

Material educativo y demostrativo, sujeto a revisión y validación en Microsoft Fabric.

# NB_06 — Corrección gramatical con `ai.fix_grammar`

**Serie:** Microsoft Fabric AI Functions con PySpark  
**Modelo de referencia:** `gpt-5-mini`  
**Entorno:** Microsoft Fabric Notebook (PySpark)

Corregiremos ortografía, gramática y puntuación de solicitudes ficticias de soporte, sin traducirlas. Revisaremos cambios, errores y consumo estimado.

**Referencia oficial:** [Use ai.fix_grammar with PySpark — Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/fix-grammar).

## 1. Requisitos y consideraciones

- Workspace en capacidad Fabric compatible y configuración de tenant habilitada.
- Sesión PySpark con AI Functions disponibles.
- Lakehouse adjunto únicamente si se desea persistir auditoría o leer archivos.
- Utilizar datos ficticios y revisar los resultados: la IA puede alterar el significado.

La función devuelve un nuevo DataFrame con el texto corregido por fila. Las entradas nulas producen resultados nulos.

## 2. Configurar el modelo y las librerías

Utilizamos el endpoint integrado de Fabric. En Runtime 2.0 (Spark 4.1), consultar la preparación temporal indicada en [configuración oficial](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration).

In [1]:
from synapse.ml.spark import aifunc

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType
)

from datetime import datetime, timezone
import uuid

MODEL_NAME = "gpt-5-mini"

aifunc.default_conf.set_deployment_name(MODEL_NAME)

print(f"Modelo configurado: {MODEL_NAME}")

StatementMeta(, cde69ddd-9bad-453c-8eff-34642519e94b, 3, Finished, Available, Finished, False)

Modelo configurado: gpt-5-mini


## 3. Referencia del método publicado por Microsoft

### Sintaxis documentada

```python
df.ai.fix_grammar(input_col="input", output_col="corrections")
```

Fuente: [Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/fix-grammar).

| Parámetro | Tipo | Requerido | Descripción |
|---|---|---|---|
| `input_col` | str | Sí | Columna existente que contiene el texto. |
| `output_col` | str | No | Nombre de la columna corregida; se genera uno si se omite. |
| `error_col` | str | No | Columna de errores; nula cuando no hay error. Se genera un nombre si se omite. |
| `input_col_type` | str | No | Usar `path` para documentos en el ejemplo multimodal. |

**Retorno:** Spark DataFrame con una columna de texto corregido. La configuración común también permite `concurrency` por llamada (predeterminado 200 por worker; máximo 1000), según [Microsoft Learn](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration). No es necesario modificarla en este laboratorio.


## 4. Crear datos ficticios

Ejemplos propios: textos en español e inglés con errores intencionales, una fila correcta y una entrada nula. No contienen datos reales.

In [2]:
data = [
    (1, "Los servidor no responde desde ayer"),
    (2, "El equipo de soporte a solucionado el problema."),
    (3, "Necesitamos revisar la configuracion del almacen de datos"),
    (4, "The reports was not updated yesterday."),
    (5, "We has a problem with the database connexion."),
    (6, "La carga de datos finalizó correctamente."),
    (7, None),
]
df = spark.createDataFrame(data, ["id", "texto"])
display(df)

StatementMeta(, cde69ddd-9bad-453c-8eff-34642519e94b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fc2ac3c0-7f62-4ffd-b10e-f185a642243d)

## 5. Ejecutar `ai.fix_grammar()`

Materializamos el resultado y lo conservamos en caché para reducir el riesgo de repetir llamadas al modelo durante la validación.

In [3]:
from pyspark import StorageLevel
run_id = str(uuid.uuid4())
resultado = df.ai.fix_grammar(
    input_col="texto",
    output_col="correccion",
    error_col="error_correccion"
)
resultado.persist(StorageLevel.MEMORY_AND_DISK)
resultado.collect()  # Dataset pequeño: materializa todas las columnas.
display(resultado)

StatementMeta(, cde69ddd-9bad-453c-8eff-34642519e94b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 079a1ba8-7270-44b7-b99e-c5c1b30dc2a5)

## 6. Validar resultados

Comparar texto original y corregido. Una modificación no implica necesariamente una mejora; verificar el significado y los términos técnicos. Una entrada nula no se interpreta como error por sí sola.

In [4]:
resultado.printSchema()
revision = resultado.withColumn(
    "estado",
    F.when(F.col("error_correccion").isNotNull(), "ERROR")
     .when(F.col("texto").isNull(), "ENTRADA_NULA")
     .when(F.col("correccion").isNull(), "SIN_RESULTADO")
     .when(F.col("texto") == F.col("correccion"), "SIN_CAMBIOS")
     .otherwise("MODIFICADO")
)
display(revision)
display(revision.groupBy("estado").count())

StatementMeta(, cde69ddd-9bad-453c-8eff-34642519e94b, 6, Finished, Available, Finished, False)

root
 |-- id: long (nullable = true)
 |-- texto: string (nullable = true)
 |-- correccion: string (nullable = true)
 |-- error_correccion: struct (nullable = true)
 |    |-- response: string (nullable = true)
 |    |-- status: struct (nullable = true)
 |    |    |-- protocolVersion: struct (nullable = true)
 |    |    |    |-- protocol: string (nullable = true)
 |    |    |    |-- major: integer (nullable = true)
 |    |    |    |-- minor: integer (nullable = true)
 |    |    |-- statusCode: integer (nullable = true)
 |    |    |-- reasonPhrase: string (nullable = true)



SynapseWidget(Synapse.DataFrame, e3849f74-275c-48c5-a597-b9d5e73e0392)

SynapseWidget(Synapse.DataFrame, f560136c-9f88-41ab-876e-bdbfba63674f)

## 8. Consultar estadísticas con `ai.stats`

Las estadísticas se consultan desde el DataFrame resultante de la operación:

```python
stats_df = resultado.ai.stats
display(stats_df)
```

La documentación incluye métricas de ejecución y consumo de tokens, entre ellas `num_successful`, `num_exceptions`, `num_unevaluated`, `input_tokens`, `cached_tokens`, `output_tokens` y `reasoning_tokens`.

La disponibilidad de cada métrica depende de la respuesta del servicio. Por eso, el código consulta los campos de forma defensiva.

In [5]:
stats_df = resultado.ai.stats

display(stats_df)

stats_rows = stats_df.collect()

if not stats_rows:
    raise RuntimeError(
        "AI Functions no devolvió estadísticas de ejecución."
    )

stats = stats_rows[0].asDict(recursive=True)

print("===== ESTADÍSTICAS AI FIX_GRAMMAR =====")
for key, value in stats.items():
    print(f"{key}: {value}")

StatementMeta(, cde69ddd-9bad-453c-8eff-34642519e94b, 7, Submitted, Running, Running, True)

## 8. Estimar el consumo de capacidad

Para `gpt-5-mini`, utilizaremos las tarifas publicadas para el endpoint integrado de Fabric:

| Tipo de token | CU-segundos por 1.000 tokens |
|---|---:|
| Entrada | 8,40 |
| Entrada en caché | 0,84 |
| Salida | 67,23 |

La fórmula es:

$$
CU_s = \\frac{(I-C)\\times 8.40 + C\\times 0.84 + O\\times 67.23}{1000}
$$

Donde:

- \(I\): tokens de entrada.
- \(C\): tokens de entrada en caché.
- \(O\): tokens de salida.

**Alcance:** el cálculo es una estimación del consumo de las llamadas al modelo integrado. No representa el consumo total de Spark ni el consumo completo de capacidad del workspace.

Los tokens de razonamiento se muestran como dato informativo y no se agregan por separado a la fórmula.

In [ ]:
if any(stats.get(k) is None for k in ("input_tokens", "output_tokens", "cached_tokens")):
    raise RuntimeError("Faltan métricas de tokens; no se puede estimar el consumo con fiabilidad.")
input_tokens = int(stats["input_tokens"])
cached_tokens = int(stats.get("cached_tokens") or 0)
output_tokens = int(stats.get("output_tokens") or 0)
reasoning_tokens = int(stats.get("reasoning_tokens") or 0)

if cached_tokens > input_tokens:
    raise ValueError(
        "cached_tokens no puede ser mayor que input_tokens."
    )

INPUT_RATE = 8.40
CACHED_RATE = 0.84
OUTPUT_RATE = 67.23

regular_input_tokens = input_tokens - cached_tokens

estimated_cu_seconds = (
    regular_input_tokens * INPUT_RATE
    + cached_tokens * CACHED_RATE
    + output_tokens * OUTPUT_RATE
) / 1000

print("===== ESTIMACIÓN DE CONSUMO =====")
print(f"Modelo reportado: {stats.get('model', MODEL_NAME)}")
print(f"Tokens de entrada: {input_tokens:,}")
print(f"Tokens en caché: {cached_tokens:,}")
print(f"Tokens de entrada normales: {regular_input_tokens:,}")
print(f"Tokens de salida: {output_tokens:,}")
print(f"Tokens de razonamiento (informativo): {reasoning_tokens:,}")
print(f"CU-segundos estimados: {estimated_cu_seconds:.6f}")

StatementMeta(, , -1, Waiting, , Waiting, True)

## 9. Construir un registro de auditoría

Se conserva el esquema de los primeros notebooks. Se registra la ejecución principal sin almacenar los textos. El estado describe la ejecución técnica, no la calidad lingüística.

In [ ]:
def metric_int(name):
    value = stats.get(name)
    return int(value) if value is not None else 0

num_input_rows = df.count()
num_successful = metric_int("num_successful")
num_exceptions = metric_int("num_exceptions")
num_unevaluated = metric_int("num_unevaluated")
num_harmful = metric_int("num_harmful")

if num_exceptions > 0 or num_harmful > 0:
    execution_status = "PARTIAL"
elif num_successful == df.filter(F.col("texto").isNotNull()).count():
    execution_status = "SUCCESS"
else:
    execution_status = "INCOMPLETE"

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("execution_timestamp_utc", StringType(), False),
    StructField("notebook_name", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("model_configured", StringType(), True),
    StructField("model_reported", StringType(), True),
    StructField("num_input_rows", LongType(), True),
    StructField("num_successful", LongType(), True),
    StructField("num_exceptions", LongType(), True),
    StructField("num_unevaluated", LongType(), True),
    StructField("num_harmful", LongType(), True),
    StructField("input_tokens", LongType(), True),
    StructField("cached_tokens", LongType(), True),
    StructField("output_tokens", LongType(), True),
    StructField("reasoning_tokens", LongType(), True),
    StructField("estimated_cu_seconds", DoubleType(), True),
    StructField("client_type", StringType(), True),
    StructField("execution_status", StringType(), True)
])

audit_record = [(
    run_id,
    datetime.now(timezone.utc).isoformat(),
    "NB_06_AI_FIX_GRAMMAR",
    "ai.fix_grammar",
    MODEL_NAME,
    str(stats.get("model") or MODEL_NAME),
    num_input_rows,
    num_successful,
    num_exceptions,
    num_unevaluated,
    num_harmful,
    input_tokens,
    cached_tokens,
    output_tokens,
    reasoning_tokens,
    float(estimated_cu_seconds),
    str(stats.get("client_type") or "unknown"),
    execution_status
)]

audit_df = spark.createDataFrame(
    audit_record,
    schema=audit_schema
)

display(audit_df)

StatementMeta(, , -1, Waiting, , Waiting, True)

## 10. Persistencia opcional en Lakehouse

Si deseas almacenar la auditoría en una tabla Delta del Lakehouse adjunto al notebook, puedes habilitar esta celda.

El modo `append` agrega un registro por ejecución. Verifica previamente el Lakehouse y el nombre de tabla.

In [ ]:
# Descomenta para persistir la auditoría en el Lakehouse adjunto.

# TABLE_NAME = "dbo.audit_ai_functions"
#
# (
#     audit_df.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(TABLE_NAME)
# )
#
# print(f"Auditoría guardada: {TABLE_NAME}")

## 11. Extensión multimodal (opcional)

Microsoft documenta la corrección de PDF y archivos de texto utilizando `input_col_type="path"`. Consultar [los requisitos multimodales](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/multimodal-overview). Este ejemplo permanece deshabilitado; genera consumo adicional y no está incluido en la auditoría principal.

In [ ]:
RUN_MULTIMODAL = False
if RUN_MULTIMODAL:
    archivos = spark.createDataFrame([("Files/ejemplo.pdf",)], ["file_path"])
    corregidos = archivos.ai.fix_grammar(
        input_col="file_path", input_col_type="path", output_col="correcciones"
    ).persist(StorageLevel.MEMORY_AND_DISK)
    corregidos.collect()
    display(corregidos)
    display(corregidos.ai.stats)
    corregidos.unpersist()

## 12. Liberar caché

In [ ]:
resultado.unpersist()

## 13. Conclusiones y referencias

Se corrigió texto por fila, se revisaron cambios y errores, y se construyó una auditoría de consumo estimado. Los CU-segundos calculados no incluyen el consumo Spark total. Verificar vigencia de tarifas antes de presupuestar.

- [Método oficial ai.fix_grammar](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/fix-grammar)
- [Configuración y estadísticas](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pyspark/configuration)
- [Facturación](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/billing)

Validación de ejecución pendiente en una capacidad real de Fabric.